In [1]:
from owslib.ogcapi.processes import Processes
import json
from pygments import highlight, lexers, formatters
import requests

In [2]:
def pprint_json(json_dict: dict) -> None:
    formatted_json = json.dumps(json_dict, sort_keys=True, indent=4)
    print(highlight(formatted_json, lexers.JsonLexer(), formatters.TerminalFormatter()))


In [3]:
SERVICE_URL = "http://localhost:4040/"


In [5]:
p = Processes(SERVICE_URL)

pprint_json(p.processes())

RuntimeError: {"type":"404","detail":"not found"}

In [47]:
pprint_json(p.process('echo'))

print("---\n")

pprint_json(p.execute('echo', inputs={'stringInput': 'foo'}))

{
    "id": "echo",
    "inputs": {
        "doubleInput": {
            "description": "This is an example of a DOUBLE literal input.",
            "schema": {
                "format": "double",
                "title": "double",
                "type": "number"
            },
            "title": "Double Literal Input Example"
        },
        "pause": {
            "description": "Optional pause duration in seconds before responding.",
            "schema": {
                "format": "uint64",
                "minimum": 0,
                "title": "uint64",
                "type": "integer"
            },
            "title": "Pause Duration"
        },
        "stringInput": {
            "description": "This is an example of a STRING literal input.",
            "schema": {
                "title": "StringInput",
                "type": "string"
            },
            "title": "String Literal Input Example"
        }
    },
    "jobControlOptions": [
        "sync-execute"

In [5]:
pprint_json(p.process('ndvi'))

{
    "id": "ndvi",
    "inputs": {
        "coordinate": {
            "description": "This is a POINT input in WGS84 (EPSG:4326) format.",
            "schema": {
                "$defs": {
                    "PointGeoJson": {
                        "properties": {
                            "coordinates": {
                                "items": {
                                    "format": "double",
                                    "type": "number"
                                },
                                "maxItems": 2,
                                "minItems": 2,
                                "type": "array"
                            },
                            "type": {
                                "$ref": "#/$defs/PointGeoJsonType"
                            }
                        },
                        "required": [
                            "type",
                            "coordinates"
                        ],
                    

In [6]:
pprint_json(p.execute('ndvi', inputs={
    "coordinate": {
        "value": {
            "type": "Point",
            "coordinates": [10.77, 49.54]
        },
        "mediaType": "application/geo+json"
    },
    "year": 2014,
    "month": 5
}))

{
    "k_ndvi": 0.26551630441320606,
    "ndvi": 0.5215686274509803
}



In [7]:
job = p.execute(
    'ndvi',
    inputs={
        "coordinate": {
            "value": {
                "type": "Point",
                "coordinates": [10.77, 49.54]
            },
            "mediaType": "application/geo+json"
        },
        "year": 2014,
        "month": 5
    },
    async_=True,
)
pprint_json(job)

job_id = job['jobID']
job_id

{
    "jobID": "ef831121-03d0-486b-97ef-16f7ce869681",
    "links": [
        {
            "href": "http://localhost:4040/jobs/ef831121-03d0-486b-97ef-16f7ce869681",
            "rel": "status",
            "title": "Job status",
            "type": "application/json"
        }
    ],
    "processID": "ndvi",
    "status": "accepted",
    "type": "process"
}



'ef831121-03d0-486b-97ef-16f7ce869681'

In [8]:
pprint_json(
    requests.get(f"{SERVICE_URL}/jobs/{job_id}").json()
)

{
    "created": "2025-11-14T12:29:37.785664Z",
    "finished": "2025-11-14T12:29:38.170090Z",
    "jobID": "ef831121-03d0-486b-97ef-16f7ce869681",
    "links": [
        {
            "href": "http://localhost:4040/jobs/ef831121-03d0-486b-97ef-16f7ce869681",
            "rel": "status",
            "title": "Job status",
            "type": "application/json"
        },
        {
            "href": "http://localhost:4040/jobs/ef831121-03d0-486b-97ef-16f7ce869681/results",
            "rel": "http://www.opengis.net/def/rel/ogc/1.0/results",
            "title": "Job result"
        }
    ],
    "message": "",
    "processID": "ndvi",
    "progress": 100,
    "status": "successful",
    "type": "process",
    "updated": "2025-11-14T12:29:38.170090Z"
}



In [9]:
pprint_json(
    requests.get(f"{SERVICE_URL}/jobs/{job_id}/results").json()
)

{
    "k_ndvi": 0.26551630441320606,
    "ndvi": 0.5215686274509803
}

